# Model Optimization: Pruning

In this notebook, we'll apply pruning techniques to our models using distributed processing. Instead of running the pruning on our notebook instance, we'll launch separate SageMaker Processing jobs to perform the pruning on more powerful instances.

## What is Pruning?

Pruning is a technique that removes unnecessary weights from a neural network, effectively making the model more sparse. Research has shown that many neural networks are overparameterized, and a significant percentage of weights can be removed without substantial impact on accuracy.

### Benefits of Pruning:
- **Reduced Model Size**: Fewer parameters means smaller models
- **Faster Inference**: Fewer computations lead to faster inference
- **Lower Memory Requirements**: Sparse models require less memory
- **Reduced Overfitting**: Removing redundant weights can improve generalization

### Types of Pruning We'll Explore:
- **Unstructured Pruning**: Removes individual weights based on their magnitude (L1 norm)
- **Structured Pruning**: Removes entire structures like neurons or channels
- **Advanced Pruning**: Uses specialized libraries for optimized pruning

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform pruning on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
import json
import time
import pandas as pd
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
import time
from IPython.display import clear_output

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Set default values that user should update
    S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Update this value
    AWS_REGION = "YOUR_REGION_HERE"      # Update this value
    SAGEMAKER_ROLE_ARN = "YOUR_ROLE_ARN_HERE"  # Update this value
    OPTIMIZATION_INSTANCE_TYPE = "ml.c5.xlarge"  # Default optimization instance type
    
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION
    %store SAGEMAKER_ROLE_ARN
    %store OPTIMIZATION_INSTANCE_TYPE

## 3. Load Model Information

In [ ]:
# Try to load model information from file
try:
    with open('model_info.json', 'r') as f:
        model_info = json.load(f)
    print(f"Loaded information for {len(model_info)} models")
except FileNotFoundError:
    print("model_info.json not found. Using default model information.")
    model_info = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased-finetuned-sst-2-english"
        }
    }

## 4. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment-analysis": "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "ner": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.",
    "question-answering": {
        "question": "What is machine learning?",
        "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
    },
    "masked-lm": "The [MASK] is a large language model trained by OpenAI."
}

## 5. Create Pruning Script

In this section, we'll create a Python script that performs the actual pruning. This script will be executed on the SageMaker Processing instances.

### What the Script Does:
1. **Loads the model and tokenizer** from Hugging Face
2. **Prepares sample inputs** for inference
3. **Applies pruning** using the specified method and amount
4. **Measures performance metrics** like model size and inference time
5. **Saves the pruned model** and metrics to the output directory

### Pruning Methods:
- **Structured Pruning**: Removes entire structures (like neurons) based on their importance
- **Advanced Pruning**: Uses specialized libraries for optimized pruning with actual size reduction
- **L1 Unstructured Pruning**: Sets individual weights to zero based on their L1 norm

The pruning amount parameter controls what percentage of weights to remove. For example, a value of 0.3 means 30% of weights will be pruned.

In [ ]:
# Check if the pruning_scripts directory exists, if not create it
import os
if not os.path.exists('pruning_scripts'):
    os.makedirs('pruning_scripts')
    print("Created pruning_scripts directory")
else:
    print("pruning_scripts directory already exists")

## 6. Launch Distributed Pruning Jobs

Now we'll set up and launch the SageMaker Processing jobs to perform pruning. Each model will be processed in a separate job, allowing for parallel processing.

### Pruning Process:
1. **Create a PyTorch processor** with the appropriate instance type and configuration
2. **For each model**:
   - Save and upload model information to S3
   - Define inputs (pruning script and model info) and outputs
   - Launch a processing job with the appropriate arguments
   - Store the job information for monitoring

We're using structured pruning with a pruning amount of 0.3 (30% of weights will be removed). This method removes entire neurons/filters, which actually reduces the model size unlike unstructured pruning.

### Parallel Processing
To speed up the process, we'll launch all jobs in parallel rather than waiting for each job to complete before starting the next one.

In [ ]:
# Define the instance type to use for pruning
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="2.0.0",
    py_version="py310",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="model-pruning",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Launch pruning jobs for all models in parallel
job_names = []  # List to store all job names
job_output_paths = {}
s3_client = boto3.client('s3')

# First, prepare all the job configurations
job_configs = {}
print("Preparing pruning jobs for all models...")

for model_key in model_info.keys():
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: model_info[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define the output path
    output_path = f's3://{S3_BUCKET}/optimization/outputs/{model_key}_pruned'
    job_output_paths[model_key] = output_path
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/data'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            output_name='pruned_model',
            source='/opt/ml/processing/output',
            destination=output_path
        )
    ]
    
    # Store the job configuration
    job_configs[model_key] = {
        'inputs': inputs,
        'outputs': outputs,
        'arguments': [
            '--model-info-path', '/opt/ml/processing/input/data/model_info.json',
            '--output-dir', '/opt/ml/processing/output',
            '--pruning-method', 'structured',  # Using structured pruning for actual size reduction
            '--pruning-amount', '0.3'
        ]
    }
    print(f"Prepared job configuration for {model_key}")

# Now launch all jobs in parallel
print("\nLaunching all pruning jobs in parallel...")
for model_key, config in job_configs.items():
    try:
        # Create a unique job name with timestamp to avoid conflicts
        timestamp = int(time.time())
        job_name = f"pruning-{model_key}-{timestamp}"
        
        # Run the processing job with the unique name
        processor.run(
            code='pruning_script.py',
            source_dir='pruning_scripts',
            inputs=config['inputs'],
            outputs=config['outputs'],
            arguments=config['arguments'],
            wait=False,  # Don't wait for the job to complete before continuing
            job_name=job_name  # Explicitly set the job name
        )
        
        # Store the job name for tracking
        job_names.append(job_name)
        print(f"Launched job for {model_key}: {job_name}")
    except Exception as e:
        print(f"Error launching job for {model_key}: {e}")

print("\nAll jobs launched. You can monitor their progress in the SageMaker console.")

In [ ]:
# Monitor job status
from sagemaker.processing import ProcessingJob

# Function to check if all jobs are complete
def are_all_jobs_complete(job_names, sagemaker_session):
    all_complete = True
    job_statuses = {}
    
    for job_name in job_names:
        job = ProcessingJob.from_processing_name(job_name, sagemaker_session)
        status = job.describe()['ProcessingJobStatus']
        job_statuses[job_name] = status
        
        if status in ['InProgress', 'Stopping']:
            all_complete = False
    
    return all_complete, job_statuses

# Poll for job completion
print("Waiting for all jobs to complete...")
while True:
    all_complete, job_statuses = are_all_jobs_complete(job_names, sagemaker_session)
    
    # Clear previous output
    clear_output(wait=True)
    
    # Print current status
    print("Current job statuses:")
    for job_name, status in job_statuses.items():
        print(f"Job {job_name}: {status}")
    
    if all_complete:
        print("All jobs completed!")
        break
    
    print("Waiting for jobs to complete... Will check again in 60 seconds.")
    time.sleep(60)  # Check every minute

print("\nAll jobs have completed or failed.")

## 7. Collect Results

Now that the pruning jobs are complete, we'll collect and combine the results from each job. Each job produces a metrics file containing information about the pruned model, such as size, inference time, and pruning parameters.

### Collection Process:
1. **Download metrics files** from S3 for each model
2. **Combine metrics** into a single dictionary
3. **Save combined metrics** to a local file for use in later notebooks

This gives us a comprehensive view of the pruning results across all models, which we'll analyze in the next section.

In [ ]:
# Download and combine results
pruned_metrics = {}
sagemaker_client = boto3.client("sagemaker")

for model_key in model_info.keys():
    # Check if we have a job for this model
    if model_key not in job_output_paths:
        print(f"No output path found for {model_key}, skipping metrics collection")
        continue
        
    # Download metrics file
    try:
        # Use the saved output path
        s3_client.download_file(
            S3_BUCKET,
            f'optimization/outputs/{model_key}_pruned/pruned_metrics.json',
            f'temp_{model_key}_pruned_metrics.json'
        )
        
        # Load metrics
        with open(f'temp_{model_key}_pruned_metrics.json', 'r') as f:
            metrics = json.load(f)
        
        # Add to combined metrics
        pruned_metrics.update(metrics)
        
        print(f"Downloaded metrics for {model_key}")
    except Exception as e:
        print(f"Error downloading metrics for {model_key}: {e}")

# Save combined metrics
with open('pruned_metrics.json', 'w') as f:
    json.dump(pruned_metrics, f, indent=2)

print(f"\nSaved pruned metrics for {len(pruned_metrics)} models to pruned_metrics.json")

## 8. Analyze Model Size Reduction

Now we'll analyze the size reduction achieved through pruning. This analysis helps us understand the impact of pruning on model size and memory footprint.

In [ ]:
# Create a DataFrame for comparison
comparison_data = []

# Function to get total size of objects with a prefix from S3
def get_total_size(bucket, prefix):
    total_size = 0
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                total_size += obj['Size']
    return total_size

for model_key in pruned_metrics.keys():
    pruned = pruned_metrics[model_key]
    
    # Get original model size from S3
    original_model_prefix = model_info[model_key]['s3_uri'].replace(f"s3://{S3_BUCKET}/", "")
    original_size = get_total_size(S3_BUCKET, original_model_prefix)
    original_size_mb = original_size / (1024 * 1024)
    
    # Get pruned model size
    pruned_model_size_mb = pruned.get('model_size', 0)
    
    # Calculate size reduction percentage
    size_reduction = ((original_size_mb - pruned_model_size_mb) / original_size_mb) * 100 if original_size_mb > 0 else 0
    
    # Prepare data for this model
    model_data = {
        'Model': pruned['model_name'],
        'Original Size (MB)': f"{original_size_mb:.2f}",
        'Pruning Method': pruned['pruning_method'],
        'Pruning Amount': f"{pruned['pruning_amount'] * 100:.1f}%",
        'Pruned Size (MB)': f"{pruned_model_size_mb:.2f}",
        'Size Reduction (%)': f"{size_reduction:.2f}",
        'Sparsity (%)': f"{pruned.get('sparsity', 0):.2f}"
    }
    
    comparison_data.append(model_data)

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display the DataFrame
comparison_df

## 9. Next Steps

Now that we've applied pruning to our models and analyzed the size reduction, we'll explore WANDA pruning in the next notebook to create even smaller, more efficient models.

### What We've Learned:
- How to apply structured pruning to transformer models for actual size reduction
- How to use SageMaker Processing for distributed optimization tasks
- How different pruning methods affect model size and memory footprint
- How to run multiple processing jobs in parallel for faster experimentation

### What's Next - WANDA Pruning:
WANDA (Weight ANalysis for Deep leArning) pruning is an advanced technique that considers both weight magnitudes and activation statistics when deciding which weights to prune. This approach tends to preserve model accuracy better than simple magnitude-based pruning.